# IMU Drift Feature Dataset — Exploratory Data Analysis

**Project**: ML-Assisted IMU Drift Reduction for GNSS-Denied UAV Navigation

**Purpose**: Thoroughly characterize the dataset before any modelling:
- Verify data integrity and provenance.
- Audit schema, quality, and sampling.
- Profile sensor, reference, and estimator signals.
- Verify target variable definitions.
- Visualize distributions, time-series behaviour, and inter-signal relationships.

**Scope limitations**: This notebook performs EDA only. No model training,
feature engineering, data cleaning, or outlier removal is applied.
Diagnostic quantities (e.g., timestamp differences) are computed for
analysis and are not part of a feature-engineering pipeline.

---
## 1. Setup and Dataset Loading

In [ ]:
import os
import sys
import hashlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid", context="notebook", palette="deep")

REPO_ROOT = Path(__file__).resolve().parent.parent if "__file__" in globals() else (
    Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
)
FIGURES_DIR = REPO_ROOT / "results" / "eda" / "figures"
TABLES_DIR  = REPO_ROOT / "results" / "eda" / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

### 1.1 Locate and load the dataset

In [ ]:
DATASET_PATH = REPO_ROOT / "data" / "imu_drift_feature_dataset.csv"
assert DATASET_PATH.exists(), f"Dataset not found at {DATASET_PATH.resolve()}"

file_size_bytes = DATASET_PATH.stat().st_size
file_size_mb = file_size_bytes / (1024 * 1024)

sha256 = hashlib.sha256()
with open(DATASET_PATH, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 16), b""):
        sha256.update(chunk)
dataset_checksum = sha256.hexdigest()

print(f"Dataset path : {DATASET_PATH.resolve()}")
print(f"File size    : {file_size_bytes:,} bytes ({file_size_mb:.2f} MB)")
print(f"SHA-256      : {dataset_checksum}")

In [ ]:
df = pd.read_csv(DATASET_PATH)
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

In [ ]:
df.head(3)

In [ ]:
df.tail(3)

In [ ]:
df.sample(3, random_state=42)

---
## 2. Schema Inspection

In [ ]:
schema_info = pd.DataFrame({
    "dtype": df.dtypes,
    "non_null": df.count(),
    "null_count": df.isnull().sum(),
    "n_unique": df.nunique(),
    "example_value": df.iloc[0],
})
schema_info["is_constant"] = schema_info["n_unique"] <= 1
print(schema_info.to_string())

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
nonnumeric_cols = df.select_dtypes(exclude="number").columns.tolist()
print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"Non-numeric columns ({len(nonnumeric_cols)}): {nonnumeric_cols}")

In [ ]:
# Constant columns
constant_cols = [c for c in df.columns if df[c].nunique() <= 1]
print(f"Constant columns: {constant_cols if constant_cols else 'None'}")

In [ ]:
schema_info.to_csv(TABLES_DIR / "schema_summary.csv")
print(f"Schema summary saved to {TABLES_DIR / 'schema_summary.csv'}")

---
## 3. Missing Values, Infinities, and Duplicates

In [ ]:
# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_report = missing_report[missing_report["missing_count"] > 0]
if missing_report.empty:
    print("No missing values found in any column.")
else:
    print(missing_report)

In [ ]:
# Infinities (positive and negative) in numeric columns
inf_report = pd.DataFrame({
    "pos_inf": (df[numeric_cols] == np.inf).sum(),
    "neg_inf": (df[numeric_cols] == -np.inf).sum(),
})
inf_report["total_inf"] = inf_report["pos_inf"] + inf_report["neg_inf"]
inf_nonzero = inf_report[inf_report["total_inf"] > 0]
if inf_nonzero.empty:
    print("No infinite values found in any numeric column.")
else:
    print(inf_nonzero)

In [ ]:
# Exact duplicate rows
n_dup_rows = df.duplicated().sum()
print(f"Exact duplicate rows: {n_dup_rows}")
if n_dup_rows > 0:
    print("First 5 duplicated rows:")
    print(df[df.duplicated(keep=False)].head(5))

In [ ]:
# Duplicate timestamps within each log
if "log_id" in df.columns and "timestamp" in df.columns:
    dup_ts = df.groupby("log_id")["timestamp"].apply(
        lambda s: s.duplicated().sum()
    )
    print("Duplicate timestamps within each log:")
    print(dup_ts.to_string())
    if dup_ts.sum() == 0:
        print("No duplicate timestamps within any log.")

In [ ]:
quality_report = pd.DataFrame({
    "missing": df.isnull().sum(),
    "pos_inf": [(df[c] == np.inf).sum() if c in numeric_cols else 0 for c in df.columns],
    "neg_inf": [(df[c] == -np.inf).sum() if c in numeric_cols else 0 for c in df.columns],
    "n_unique": df.nunique(),
})
quality_report.to_csv(TABLES_DIR / "quality_audit.csv")
print(f"Quality audit saved to {TABLES_DIR / 'quality_audit.csv'}")

---
## 4. Recording Groups and Timestamp Analysis

In [ ]:
logs = df["log_id"].unique()
print(f"Number of logs: {len(logs)}")
for lid in logs:
    print(f"  {lid}")

In [ ]:
log_summary_rows = []
for lid in logs:
    grp = df[df["log_id"] == lid].copy()
    ts = grp["timestamp"].values
    dt_us = np.diff(ts)  # microsecond differences

    time_s = grp["time_s"].values
    elapsed_ts = (ts - ts[0]) / 1e6  # convert µs to seconds
    elapsed_time_s = time_s - time_s[0]

    row = {
        "log_id": lid,
        "n_rows": len(grp),
        "timestamp_min": ts[0],
        "timestamp_max": ts[-1],
        "time_s_min": time_s[0],
        "time_s_max": time_s[-1],
        "time_span_s": (ts[-1] - ts[0]) / 1e6,
        "is_monotonic": bool(np.all(dt_us > 0)),
        "n_repeated_ts": int(np.sum(dt_us == 0)),
        "n_decreasing_ts": int(np.sum(dt_us < 0)),
        "dt_min_us": int(dt_us.min()) if len(dt_us) > 0 else None,
        "dt_median_us": int(np.median(dt_us)) if len(dt_us) > 0 else None,
        "dt_p90_us": int(np.percentile(dt_us, 90)) if len(dt_us) > 0 else None,
        "dt_p99_us": int(np.percentile(dt_us, 99)) if len(dt_us) > 0 else None,
        "dt_max_us": int(dt_us.max()) if len(dt_us) > 0 else None,
        "n_gaps_over_10ms": int(np.sum(dt_us > 10_000)),
        "max_elapsed_diff": float(np.max(np.abs(elapsed_ts - elapsed_time_s))),
    }
    log_summary_rows.append(row)

log_summary = pd.DataFrame(log_summary_rows)
print(log_summary.to_string(index=False))

In [ ]:
log_summary.to_csv(TABLES_DIR / "log_summary.csv", index=False)
print(f"Log summary saved to {TABLES_DIR / 'log_summary.csv'}")

### 4.1 Timestamp vs time_s consistency

Both `timestamp` (integer, likely microseconds) and `time_s` (float, seconds)
exist. We test whether `time_s` equals `(timestamp - timestamp_0) / 1e6`
within each log.

In [ ]:
for lid in logs:
    grp = df[df["log_id"] == lid]
    ts = grp["timestamp"].values.astype(np.float64)
    time_s = grp["time_s"].values
    elapsed_from_ts = (ts - ts[0]) / 1e6
    diff = np.abs(elapsed_from_ts - (time_s - time_s[0]))
    print(f"{lid}: max |elapsed_ts - elapsed_time_s| = {diff.max():.6e} s, "
          f"mean = {diff.mean():.6e} s")

### 4.2 Sampling interval distributions

The CSV row cadence (time between successive rows) is distinct from the
native sensor sampling rate. The sensor may sample at a higher rate and be
downsampled or merged during extraction. The nominal ~5 ms interval
corresponds to ~200 Hz row cadence.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.ravel()
for i, lid in enumerate(logs):
    grp = df[df["log_id"] == lid]
    ts = grp["timestamp"].values
    dt_ms = np.diff(ts) / 1e3  # microseconds to milliseconds
    ax = axes[i]
    ax.hist(dt_ms, bins=100, edgecolor="k", alpha=0.7)
    ax.set_title(f"{lid}", fontsize=10)
    ax.set_xlabel("Interval (ms)")
    ax.set_ylabel("Count")
    ax.axvline(5.0, color="red", linestyle="--", alpha=0.7, label="5 ms")
    ax.legend(fontsize=8)
fig.suptitle("Sampling Interval Distributions per Log", fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "sampling_intervals.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Locations of large gaps (> 10 ms)
for lid in logs:
    grp = df[df["log_id"] == lid].reset_index(drop=True)
    ts = grp["timestamp"].values
    dt_us = np.diff(ts)
    gap_idx = np.where(dt_us > 10_000)[0]
    if len(gap_idx) > 0:
        print(f"\n{lid}: {len(gap_idx)} gaps > 10 ms")
        for idx in gap_idx[:10]:  # show first 10
            print(f"  Row {idx}→{idx+1}: dt = {dt_us[idx]/1e3:.2f} ms, "
                  f"time_s = {grp.at[idx, 'time_s']:.4f}")
    else:
        print(f"{lid}: No gaps > 10 ms")

---
## 5. IMU Channel Profiling

In [ ]:
accel_cols = ["accel_x_m_s2", "accel_y_m_s2", "accel_z_m_s2"]
gyro_cols = ["gyro_x_rad_s", "gyro_y_rad_s", "gyro_z_rad_s"]
imu_cols = accel_cols + gyro_cols

imu_stats = df[imu_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
print(imu_stats.to_string())

In [ ]:
imu_stats.to_csv(TABLES_DIR / "imu_statistics.csv")
print(f"IMU statistics saved to {TABLES_DIR / 'imu_statistics.csv'}")

### 5.1 Accelerometer magnitude verification

The column `accel_magnitude_m_s2` should equal
`sqrt(accel_x² + accel_y² + accel_z²)`.

In [ ]:
computed_accel_mag = np.sqrt(
    df["accel_x_m_s2"]**2 + df["accel_y_m_s2"]**2 + df["accel_z_m_s2"]**2
)
accel_mag_diff = np.abs(df["accel_magnitude_m_s2"] - computed_accel_mag)
print(f"Accelerometer magnitude verification:")
print(f"  Max |stored - computed|  = {accel_mag_diff.max():.2e}")
print(f"  Mean |stored - computed| = {accel_mag_diff.mean():.2e}")
print(f"  Match within 1e-6: {(accel_mag_diff < 1e-6).all()}")

### 5.2 Gyroscope magnitude verification

The column `gyro_magnitude_rad_s` should equal
`sqrt(gyro_x² + gyro_y² + gyro_z²)`.

In [ ]:
computed_gyro_mag = np.sqrt(
    df["gyro_x_rad_s"]**2 + df["gyro_y_rad_s"]**2 + df["gyro_z_rad_s"]**2
)
gyro_mag_diff = np.abs(df["gyro_magnitude_rad_s"] - computed_gyro_mag)
print(f"Gyroscope magnitude verification:")
print(f"  Max |stored - computed|  = {gyro_mag_diff.max():.2e}")
print(f"  Mean |stored - computed| = {gyro_mag_diff.mean():.2e}")
print(f"  Match within 1e-6: {(gyro_mag_diff < 1e-6).all()}")

### 5.3 Physical interpretation notes

- **Units**: Column names suggest `m/s²` for accelerometers and `rad/s` for
  gyroscopes. These are consistent with PX4 `sensor_combined` topic units.
- **Coordinate frame**: Unknown. PX4 typically uses FRD (front-right-down)
  body frame for IMU data, but this cannot be confirmed from the CSV alone.
- **Gravity**: At rest, we would expect `accel_z ≈ -9.81 m/s²` in FRD.
  The mean of `accel_z_m_s2` is approximately -9.78, consistent with a
  body-frame z-axis reading including gravity. This suggests raw
  (not gravity-compensated) accelerometer data.
- **Calibration**: No information about calibration status is available.

---
## 6. GPS, PX4 Estimator, and Reference Fields

In [ ]:
gps_vel_cols = ["gps_vel_n_m_s", "gps_vel_e_m_s", "gps_vel_d_m_s"]
gps_quality_cols = ["vel_ned_valid", "fix_type", "gps_speed_m_s",
                    "gps_eph_m", "gps_epv_m", "gps_satellites_used"]
px4_vel_cols = ["px4_vx_m_s", "px4_vy_m_s", "px4_vz_m_s"]
px4_validity_cols = ["v_xy_valid", "v_z_valid", "xy_valid", "z_valid"]
dr_cols = ["dead_reckoning"]

print("=== GPS Velocity Statistics ===")
print(df[gps_vel_cols].describe().T.to_string())

In [ ]:
print("\n=== GPS Quality Fields ===")
print(df[gps_quality_cols].describe().T.to_string())

In [ ]:
print("\n=== Unique values in GPS quality / validity fields ===")
for c in gps_quality_cols + ["vel_ned_valid"]:
    print(f"  {c}: {sorted(df[c].unique())}")

In [ ]:
print("\n=== PX4 Velocity Statistics ===")
print(df[px4_vel_cols].describe().T.to_string())

In [ ]:
print("\n=== PX4 Validity Flags ===")
for c in px4_validity_cols + dr_cols:
    print(f"  {c}: unique = {sorted(df[c].unique())}")

### 6.1 Absent columns

The following categories are **not present** in this dataset:
- GPS position (lat, lon, alt).
- PX4 position estimates.
- Attitude (quaternion / Euler angles).
- Magnetometer readings.
- Barometer readings.

### 6.2 Repeated consecutive values in GPS and PX4 velocity

GPS receivers typically update at a lower rate than the IMU (e.g., 5–10 Hz
vs 200 Hz). Between updates, the same GPS velocity may be repeated in merged
rows. Similarly, the PX4 estimator may update at a different cadence.

We count runs of consecutive identical values. **Note**: Observing value
changes does not confirm a new sensor measurement arrived at that exact row;
it only indicates the merged value changed.

In [ ]:
for col_group, label in [(gps_vel_cols, "GPS velocity"), (px4_vel_cols, "PX4 velocity")]:
    print(f"\n=== Repeated consecutive values: {label} ===")
    for lid in logs:
        grp = df[df["log_id"] == lid]
        changes = {}
        for c in col_group:
            vals = grp[c].values
            n_changes = np.sum(vals[1:] != vals[:-1])
            changes[c] = n_changes
        total_rows = len(grp) - 1
        first_col = col_group[0]
        print(f"  {lid}: {changes[first_col]} value changes / {total_rows} intervals "
              f"({changes[first_col]/total_rows*100:.1f}%) for {first_col}")

---
## 7. Target Definition Verification

In [ ]:
target_cols = ["target_delta_v_n_m_s", "target_delta_v_e_m_s", "target_delta_v_d_m_s"]

# Verify: target = px4_velocity - gps_velocity
computed_n = df["px4_vx_m_s"] - df["gps_vel_n_m_s"]
computed_e = df["px4_vy_m_s"] - df["gps_vel_e_m_s"]
computed_d = df["px4_vz_m_s"] - df["gps_vel_d_m_s"]

diff_n = np.abs(df["target_delta_v_n_m_s"] - computed_n)
diff_e = np.abs(df["target_delta_v_e_m_s"] - computed_e)
diff_d = np.abs(df["target_delta_v_d_m_s"] - computed_d)

target_verif = pd.DataFrame({
    "component": ["north (n)", "east (e)", "down (d)"],
    "max_abs_diff": [diff_n.max(), diff_e.max(), diff_d.max()],
    "mean_abs_diff": [diff_n.mean(), diff_e.mean(), diff_d.mean()],
    "match_within_1e-10": [
        (diff_n < 1e-10).all(),
        (diff_e < 1e-10).all(),
        (diff_d < 1e-10).all(),
    ],
})
print("Target verification: target_delta_v = px4_v - gps_vel")
print(target_verif.to_string(index=False))

In [ ]:
target_verif.to_csv(TABLES_DIR / "target_verification.csv", index=False)
print(f"Target verification saved to {TABLES_DIR / 'target_verification.csv'}")

In [ ]:
print("\n=== Target statistics ===")
print(df[target_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T.to_string())

---
## 8. Distributions and Potential Outliers

In [ ]:
# Global descriptive statistics for all numeric columns
desc_global = df[numeric_cols].describe(
    percentiles=[0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
).T
desc_global.to_csv(TABLES_DIR / "descriptive_statistics_global.csv")
print(f"Global statistics saved ({len(desc_global)} columns)")

In [ ]:
# Per-log statistics for selected columns
selected_cols = imu_cols + target_cols + ["gps_eph_m", "gps_satellites_used"]
per_log_stats = df.groupby("log_id")[selected_cols].describe(
    percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]
)
per_log_stats.to_csv(TABLES_DIR / "descriptive_statistics_per_log.csv")
print(f"Per-log statistics saved ({per_log_stats.shape})")

### 8.1 Potential outliers using 1.5×IQR rule

We use the classic 1.5×IQR rule as a diagnostic to flag values that lie
far from the central distribution. These are flagged as **potential
outliers requiring investigation**, not as confirmed faulty measurements.
Extreme IMU readings can be physically real (e.g., during aggressive
manoeuvres or vibration).

In [ ]:
outlier_summary_rows = []
for c in imu_cols + target_cols:
    q1 = df[c].quantile(0.25)
    q3 = df[c].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (df[c] < lower) | (df[c] > upper)
    n_outliers = mask.sum()
    outlier_summary_rows.append({
        "column": c,
        "Q1": q1, "Q3": q3, "IQR": iqr,
        "lower_fence": lower, "upper_fence": upper,
        "n_flagged": n_outliers,
        "pct_flagged": round(n_outliers / len(df) * 100, 2),
    })

outlier_summary = pd.DataFrame(outlier_summary_rows)
print(outlier_summary.to_string(index=False))

In [ ]:
outlier_summary.to_csv(TABLES_DIR / "outlier_flags_iqr.csv", index=False)
print(f"Outlier flags saved to {TABLES_DIR / 'outlier_flags_iqr.csv'}")

In [ ]:
# Notable extremes with log_id and time_s
extreme_cols = ["accel_magnitude_m_s2", "gyro_magnitude_rad_s"] + target_cols
for c in extreme_cols:
    idx_max = df[c].idxmax()
    idx_min = df[c].idxmin()
    print(f"\n{c}:")
    print(f"  MAX = {df.at[idx_max, c]:.4f} at log={df.at[idx_max, 'log_id']}, "
          f"time_s={df.at[idx_max, 'time_s']:.4f}")
    print(f"  MIN = {df.at[idx_min, c]:.4f} at log={df.at[idx_min, 'log_id']}, "
          f"time_s={df.at[idx_min, 'time_s']:.4f}")

---
## 9. Temporal Visualizations

### 9.1 Accelerometer time series

In [ ]:
fig, axes = plt.subplots(len(logs), 1, figsize=(16, 3.5 * len(logs)), sharex=False)
if len(logs) == 1:
    axes = [axes]
for i, lid in enumerate(logs):
    grp = df[df["log_id"] == lid]
    ax = axes[i]
    for c, color in zip(accel_cols, ["#e74c3c", "#27ae60", "#2980b9"]):
        ax.plot(grp["time_s"], grp[c], label=c.replace("accel_", "").replace("_m_s2", ""),
                alpha=0.7, linewidth=0.5, color=color)
    ax.set_title(f"Accelerometer — {lid}", fontsize=10)
    ax.set_ylabel("m/s²")
    ax.legend(loc="upper right", fontsize=8)
axes[-1].set_xlabel("time_s")
fig.suptitle("Accelerometer Axes per Log", fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "accel_time_series.png", dpi=150, bbox_inches="tight")
plt.show()

### 9.2 Gyroscope time series

In [ ]:
fig, axes = plt.subplots(len(logs), 1, figsize=(16, 3.5 * len(logs)), sharex=False)
if len(logs) == 1:
    axes = [axes]
for i, lid in enumerate(logs):
    grp = df[df["log_id"] == lid]
    ax = axes[i]
    for c, color in zip(gyro_cols, ["#e74c3c", "#27ae60", "#2980b9"]):
        ax.plot(grp["time_s"], grp[c], label=c.replace("gyro_", "").replace("_rad_s", ""),
                alpha=0.7, linewidth=0.5, color=color)
    ax.set_title(f"Gyroscope — {lid}", fontsize=10)
    ax.set_ylabel("rad/s")
    ax.legend(loc="upper right", fontsize=8)
axes[-1].set_xlabel("time_s")
fig.suptitle("Gyroscope Axes per Log", fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "gyro_time_series.png", dpi=150, bbox_inches="tight")
plt.show()

### 9.3 GPS vs PX4 velocity

In [ ]:
vel_pairs = [
    ("gps_vel_n_m_s", "px4_vx_m_s", "North / X"),
    ("gps_vel_e_m_s", "px4_vy_m_s", "East / Y"),
    ("gps_vel_d_m_s", "px4_vz_m_s", "Down / Z"),
]
fig, axes = plt.subplots(3, len(logs), figsize=(5 * len(logs), 10), sharex=False)
if len(logs) == 1:
    axes = axes.reshape(-1, 1)
for j, lid in enumerate(logs):
    grp = df[df["log_id"] == lid]
    for i, (gps_c, px4_c, label) in enumerate(vel_pairs):
        ax = axes[i, j]
        ax.plot(grp["time_s"], grp[gps_c], alpha=0.7, linewidth=0.5,
                label="GPS", color="#3498db")
        ax.plot(grp["time_s"], grp[px4_c], alpha=0.7, linewidth=0.5,
                label="PX4", color="#e67e22")
        if i == 0:
            ax.set_title(lid, fontsize=9)
        if j == 0:
            ax.set_ylabel(f"{label} (m/s)")
        ax.legend(fontsize=7, loc="upper right")
axes[-1, 0].set_xlabel("time_s")
fig.suptitle("GPS vs PX4 Velocity Components per Log", fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "gps_vs_px4_velocity.png", dpi=150, bbox_inches="tight")
plt.show()

### 9.4 Target residuals over time

In [ ]:
fig, axes = plt.subplots(len(logs), 1, figsize=(16, 3.5 * len(logs)), sharex=False)
if len(logs) == 1:
    axes = [axes]
for i, lid in enumerate(logs):
    grp = df[df["log_id"] == lid]
    ax = axes[i]
    for c, color in zip(target_cols, ["#9b59b6", "#1abc9c", "#e74c3c"]):
        ax.plot(grp["time_s"], grp[c], alpha=0.7, linewidth=0.5,
                label=c.replace("target_delta_v_", "Δv_").replace("_m_s", ""),
                color=color)
    ax.set_title(f"Target Residuals — {lid}", fontsize=10)
    ax.set_ylabel("m/s")
    ax.legend(loc="upper right", fontsize=8)
    ax.axhline(0, color="gray", linestyle=":", alpha=0.5)
axes[-1].set_xlabel("time_s")
fig.suptitle("Velocity Residuals (PX4 − GPS) per Log", fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "target_residuals_time_series.png", dpi=150, bbox_inches="tight")
plt.show()

### 9.5 Sampling intervals over time

In [ ]:
fig, axes = plt.subplots(len(logs), 1, figsize=(16, 2.5 * len(logs)), sharex=False)
if len(logs) == 1:
    axes = [axes]
for i, lid in enumerate(logs):
    grp = df[df["log_id"] == lid]
    ts = grp["timestamp"].values
    dt_ms = np.diff(ts) / 1e3
    ax = axes[i]
    ax.plot(grp["time_s"].values[1:], dt_ms, linewidth=0.3, alpha=0.7, color="#2c3e50")
    ax.axhline(5.0, color="red", linestyle="--", alpha=0.5, label="5 ms")
    ax.set_title(f"Row Intervals — {lid}", fontsize=10)
    ax.set_ylabel("Interval (ms)")
    ax.legend(fontsize=8)
axes[-1].set_xlabel("time_s")
fig.suptitle("Sampling Intervals Over Time", fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "sampling_intervals_over_time.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 10. Distribution and Recording Comparisons

### 10.1 IMU channel distributions by log

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for i, c in enumerate(imu_cols):
    ax = axes[i // 3, i % 3]
    for lid in logs:
        grp = df[df["log_id"] == lid]
        ax.hist(grp[c], bins=80, alpha=0.5, label=lid[:6], density=True)
    ax.set_title(c, fontsize=10)
    ax.set_ylabel("Density")
    ax.legend(fontsize=7)
fig.suptitle("IMU Channel Distributions by Log", fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "imu_distributions_by_log.png", dpi=150, bbox_inches="tight")
plt.show()

### 10.2 Target distributions by log

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for i, c in enumerate(target_cols):
    ax = axes[i]
    for lid in logs:
        grp = df[df["log_id"] == lid]
        ax.hist(grp[c], bins=80, alpha=0.5, label=lid[:6], density=True)
    ax.set_title(c.replace("target_delta_v_", "Δv_"), fontsize=10)
    ax.set_ylabel("Density")
    ax.legend(fontsize=7)
fig.suptitle("Target Residual Distributions by Log", fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "target_distributions_by_log.png", dpi=150, bbox_inches="tight")
plt.show()

### 10.3 Box plots comparing logs

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, c in enumerate(target_cols):
    ax = axes[i]
    data_by_log = [df[df["log_id"] == lid][c].values for lid in logs]
    try:
        bp = ax.boxplot(data_by_log, tick_labels=[lid[:6] for lid in logs], patch_artist=True)
    except TypeError:
        bp = ax.boxplot(data_by_log, labels=[lid[:6] for lid in logs], patch_artist=True)
    colors = ["#3498db", "#e74c3c", "#2ecc71", "#f39c12"]
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_title(c.replace("target_delta_v_", "Δv_"), fontsize=10)
    ax.set_ylabel("m/s")
fig.suptitle("Target Residual Box Plots by Log", fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "target_boxplots_by_log.png", dpi=150, bbox_inches="tight")
plt.show()

### 10.4 Per-log summary comparison

Differences across logs are described statistically. Without supporting
metadata (e.g., flight mode, wind conditions, pilot inputs), we cannot
attribute differences to specific flight conditions such as hovering
or aggressive manoeuvres.

In [ ]:
per_log_mean = df.groupby("log_id")[imu_cols + target_cols].mean()
print(per_log_mean.T.to_string())
per_log_mean.to_csv(TABLES_DIR / "per_log_means.csv")

---
## 11. Correlations and Temporal Dependence

### 11.1 Correlation matrix

We compute Pearson correlations between selected numeric signals,
excluding identifiers (`log_id`, `timestamp`) and constant/near-constant
validity flags.

In [ ]:
corr_cols = imu_cols + ["accel_magnitude_m_s2", "gyro_magnitude_rad_s"] + \
            gps_vel_cols + px4_vel_cols + target_cols + \
            ["gps_eph_m", "gps_epv_m", "gps_satellites_used"]

corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", center=0,
            cmap="RdBu_r", vmin=-1, vmax=1, square=True, linewidths=0.5,
            annot_kws={"size": 7}, ax=ax)
ax.set_title("Pearson Correlation Matrix (Selected Signals)", fontsize=13)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

**Interpretation note**: Correlations involving target components
(`target_delta_v_*`) with PX4 or GPS velocity components can be
algebraically induced because `target = px4 - gps`. High correlations
here do not constitute evidence of causal sensor-error relationships.

### 11.2 Sensor vs target scatter plots

In [ ]:
scatter_pairs = [
    ("accel_x_m_s2", "target_delta_v_n_m_s"),
    ("accel_y_m_s2", "target_delta_v_e_m_s"),
    ("accel_z_m_s2", "target_delta_v_d_m_s"),
    ("gyro_magnitude_rad_s", "target_delta_v_n_m_s"),
    ("gps_eph_m", "target_delta_v_n_m_s"),
    ("gps_satellites_used", "target_delta_v_n_m_s"),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()
for i, (x_col, y_col) in enumerate(scatter_pairs):
    ax = axes[i]
    # Subsample for readability
    sample = df.sample(min(5000, len(df)), random_state=42)
    ax.scatter(sample[x_col], sample[y_col], alpha=0.2, s=5, color="#2c3e50")
    ax.set_xlabel(x_col, fontsize=9)
    ax.set_ylabel(y_col, fontsize=9)
    r = df[x_col].corr(df[y_col])
    ax.set_title(f"r = {r:.3f}", fontsize=10)
fig.suptitle("Sensor vs Target Scatter Plots (sampled)", fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "sensor_vs_target_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

### 11.3 Autocorrelation of target residuals

Strong autocorrelation in target signals indicates temporal dependence:
successive rows carry dependent information. This means random row-wise
splitting for train/test would leak future information into training,
producing artificially optimistic error estimates.

In [ ]:
fig, axes = plt.subplots(len(logs), 3, figsize=(16, 3.5 * len(logs)))
if len(logs) == 1:
    axes = axes.reshape(1, -1)
max_lags = 200

for i, lid in enumerate(logs):
    grp = df[df["log_id"] == lid]
    # Only use continuous segments (no large gaps)
    ts = grp["timestamp"].values
    dt_us = np.diff(ts)
    # Find longest continuous segment (all intervals < 10 ms)
    gap_idx = np.where(dt_us > 10_000)[0]
    if len(gap_idx) > 0:
        boundaries = np.concatenate([[-1], gap_idx, [len(grp) - 1]])
        seg_lengths = np.diff(boundaries)
        best_seg = np.argmax(seg_lengths)
        start = boundaries[best_seg] + 1
        end = boundaries[best_seg + 1] + 1
    else:
        start, end = 0, len(grp)
    seg = grp.iloc[start:end]

    for j, c in enumerate(target_cols):
        ax = axes[i, j]
        vals = seg[c].values
        if len(vals) > max_lags + 1:
            autocorr = [np.corrcoef(vals[:-lag], vals[lag:])[0, 1] for lag in range(1, max_lags + 1)]
            ax.bar(range(1, max_lags + 1), autocorr, width=1.0, alpha=0.7, color="#2980b9")
        ax.set_ylim(-0.2, 1.05)
        ax.axhline(0, color="gray", linestyle=":", alpha=0.5)
        if i == 0:
            ax.set_title(c.replace("target_delta_v_", "Δv_"), fontsize=10)
        if j == 0:
            ax.set_ylabel(f"{lid[:6]}\nAutocorr")
        if i == len(logs) - 1:
            ax.set_xlabel("Lag (rows)")

fig.suptitle("Autocorrelation of Target Residuals (longest continuous segment per log)",
             fontsize=12, y=1.01)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "target_autocorrelation.png", dpi=150, bbox_inches="tight")
plt.show()

**Key implication**: If autocorrelation remains high for tens or hundreds
of lags, a random row-wise train/test split would allow near-identical
rows to appear in both sets. Proper evaluation requires time-based or
log-based splitting strategies.

---
## 12. Summary

Key findings are documented in `docs/eda_report.md`. The data dictionary
is in `docs/data_dictionary.md` and target definitions in
`docs/target_definition.md`.

This notebook should be run from a clean kernel in order to reproduce all
results. Figures are saved to `results/eda/figures/` and tables to
`results/eda/tables/`.

In [ ]:
# Final checksum verification — confirm dataset was not modified
sha256_final = hashlib.sha256()
with open(DATASET_PATH, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 16), b""):
        sha256_final.update(chunk)
final_checksum = sha256_final.hexdigest()

print(f"Original checksum: {dataset_checksum}")
print(f"Final checksum:    {final_checksum}")
print(f"Dataset unchanged: {dataset_checksum == final_checksum}")